# Graph Positional and Structural Embeddings (GPSE)

Graph Representation on ZINC: Learning rich positional and structural node features for expressive GNNs. This notebook implements the approach with `GPSE` inside a `K3GPSE` model, trained with the Adam optimizer, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `GPSE` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers

title = "Graph Positional and Structural Encoder (GPSE)"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. GPSE Model Definition
class K3GPSE(keras.Model):
    def __init__(self, in_channels, pe_dim, hidden_channels, out_channels):
        super().__init__()
        self.node_emb = layers.Dense(hidden_channels)
        self.pe_emb = layers.Dense(hidden_channels)
        self.conv1 = k3_layers.GCNConv(hidden_channels, hidden_channels)
        self.conv2 = k3_layers.GCNConv(hidden_channels, out_channels)

    def call(self, x, pe, edge_index):
        h = self.node_emb(x) + self.pe_emb(pe)
        h = ops.relu(self.conv1(h, edge_index))
        return self.conv2(h, edge_index)

k3_model = K3GPSE(in_channels=16, pe_dim=8, hidden_channels=64, out_channels=2)

# 2. Eager Verification
num_nodes = 50
dummy_x = keras.random.normal((num_nodes, 16))
dummy_pe = keras.random.normal((num_nodes, 8))
dummy_edges = ops.convert_to_tensor([[0, 1], [1, 0]], dtype="int64")

out = k3_model(dummy_x, dummy_pe, dummy_edges)
print(f"Forward pass successful! Output shape: {out.shape} (Expected: ({num_nodes}, 2))")

k3_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.MeanSquaredError(),
)
print("GPSE model compiled successfully!")

print("\n✓ K3-Node GPSE execution completed successfully!")